# 课堂教学案例：西财新闻怎么拉下来

> 给谁：W3 第 3 节。  
> 数据：[西财要闻](https://www.swufe.edu.cn/index/xcyw.htm)  
> 工具：`requests` 下载 HTML，`BeautifulSoup` 从 HTML 里取出标题、日期、链接、正文。  
> 当堂：跑通「列表首页 + 1 篇正文」。翻完全站、下完全文留课后（必须间隔请求，不要把学校网站打挂）。  


## 0. 先想清楚：分析西财近些年新闻，该如何做

不要一上来就写爬虫。先问四件事（对应 `data-taxonomy.md`）：

| 问 | 这份新闻大概是 |
|---|---|
| 从哪来 | **客观**（学校发过的稿），不是合成 |
| 谁能看 | **公开**（官网，不用登录） |
| 花不花钱 | **免费** |
| 怎么拿 | **网页通道**（没有现成 API，HTML 里就有） |
| 长什么样 | 列表是半结构化；正文是 **非结构化文本** |
| 能不能进产品 | 学习、分析可以；转载原文要看学校规定，默认只存自己用 |

分析路径也先写在纸上，再动手：

1. **列表**：每条新闻的日期、标题、链接
2. **正文**：点进去拿到完整段落
3. **落盘**：csv / jsonl，方便以后按年统计、做词频、选题
4. **分析**（本周不必做完）：近年发稿量、主题词、哪些学院常出现——那是有了数据之后的事

本页只做到第 3 步。网页采集的系统讲法见 W4 的 `knowledge/phase1-1-4b-web-data-collection.md`。

环境（在项目 `.venv` 里）：

```text
python -m pip install requests beautifulsoup4 pandas
```


## 1. 两个工具各干什么

打开网页，浏览器其实做了两件事：向服务器要一份 HTML 文件，再把文件画成你看见的页面。

- **`requests`**：替你「要文件」。给一个网址，返回状态码和文本。它**不理解**标题在哪。
- **`BeautifulSoup`**：替你「在这份 HTML 里找标签」。比如所有 `h3`、所有带 `class="publine"` 的链接。

口诀：**requests 负责拿，BeautifulSoup 负责拆。** 先拿后拆。编码不对会乱码，西财这站用 UTF-8：`resp.encoding = "utf-8"`。

礼貌：学校官网 `robots.txt` 当前是 404，仍要当客人——加 `User-Agent`、设 `timeout`、翻页时 `time.sleep`，课堂不要开多线程狂刷。


## 2. 浅：先把首页 HTML 拿到手


In [3]:
import requests

URL = "https://www.swufe.edu.cn/index/xcyw.htm"
headers = {"User-Agent": "SWUFE-DA-class/0.1 (teaching; polite crawl)"}

resp = requests.get(URL, headers=headers, timeout=15)
# resp.encoding = "utf-8"
print(resp.status_code)      # 200 才算拿到
print(resp.text[:400])       # 先看是不是 HTML，不要假设


200
ï»¿<!DOCTYPE html>
<html lang="zh-CN">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0,minimal-ui:ios">
    <meta http-equiv="X-UA-Compatible" content="ie=edge">
    <title>è¥¿è´¢è¦é»-è¥¿åè´¢ç»å¤§å­¦</title><meta name="pageType" content="2">
<meta name="pageTitle" content="è¥¿è´¢è¦é»">
<


## 2.1 encoding 和 decoding 不是一回事

一对方向相反的操作，课上最容易混：

| 词 | 中文 | 在干什么 |
|---|---|---|
| **encoding** | 编码 | 字 → 字节。服务器把「西财要闻」按 UTF-8 **编成**一串字节再发给你。 |
| **decoding** | 解码 | 字节 → 字。你这边必须用**同一套规则**把字节 **解回**汉字。 |

网页传过来的是**字节**（`resp.content`），不是你看见的字（`resp.text`）。编的时候用 UTF-8，解的时候却用 Latin-1 / `ISO-8859-1`，就会乱码。`requests` 头里常读不到 charset，有时会用错解码；所以要写 `resp.encoding = "utf-8"`——告诉它：**按 UTF-8 做 decoding**。西财页面 `<meta charset="UTF-8">` 已经声明了。

口诀：**怎么 encode 的，就怎么 decode。** 西财这站两端都是 UTF-8。

下面用已经拿到的 `resp.content` 对比两种 decoding，不再请求一次。乱码那行是预期现象。 



In [4]:
resp.content

b'\xef\xbb\xbf<!DOCTYPE html>\r\n<html lang="zh-CN">\r\n\r\n<head>\r\n    <meta charset="UTF-8">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0,minimal-ui:ios">\r\n    <meta http-equiv="X-UA-Compatible" content="ie=edge">\r\n    <title>\xe8\xa5\xbf\xe8\xb4\xa2\xe8\xa6\x81\xe9\x97\xbb-\xe8\xa5\xbf\xe5\x8d\x97\xe8\xb4\xa2\xe7\xbb\x8f\xe5\xa4\xa7\xe5\xad\xa6</title><meta name="pageType" content="2">\r\n<meta name="pageTitle" content="\xe8\xa5\xbf\xe8\xb4\xa2\xe8\xa6\x81\xe9\x97\xbb">\r\n<META Name="keywords" Content="\xe8\xa5\xbf\xe5\x8d\x97\xe8\xb4\xa2\xe7\xbb\x8f\xe5\xa4\xa7\xe5\xad\xa6,\xe8\xa5\xbf\xe8\xb4\xa2\xe6\x96\xb0\xe9\x97\xbb" />\r\n\r\n    <link rel="stylesheet" href="../style/reset.css">\r\n    <link rel="stylesheet" href="../style/public.css">\r\n    <link rel="stylesheet" href="../style/ny2.css">\r\n    <link rel="stylesheet" href="../style/ny_media2.css">\r\n\r\n\r\n<!--Announced by Visual SiteBuilder 9-->\r\

In [6]:
# encoding：字→字节（对方已经编好了，在 content 里）
# decoding：字节→字（我们这边 decode）
print("告诉 requests 用哪种 decoding:", resp.encoding)
print("库猜的 apparent_encoding:", resp.apparent_encoding)

wrong = resp.content.decode("latin-1")   # 错的 decoding
right = resp.content.decode("utf-8")     # 和网站 encoding 一致
print("\n用 latin-1 解码（乱码，预期如此）:\n", wrong[180:320])
print("\n用 UTF-8 解码（可读）:\n", right[180:320])

告诉 requests 用哪种 decoding: ISO-8859-1
库猜的 apparent_encoding: UTF-8-SIG

用 latin-1 解码（乱码，预期如此）:
 lable=0,minimal-ui:ios">
    <meta http-equiv="X-UA-Compatible" content="ie=edge">
    <title>è¥¿è´¢è¦é»-è¥¿åè´¢ç»å¤§å­¦</title><meta

用 UTF-8 解码（可读）:
 ble=0,minimal-ui:ios">
    <meta http-equiv="X-UA-Compatible" content="ie=edge">
    <title>西财要闻-西南财经大学</title><meta name="pageType" conte


`200` 再往下走。不是 200：把状态码记进可得性初验，换源或下课再试。


## 3. 拆列表：日期、标题、链接

用浏览器「检查」会看到每条新闻是：

```html
<a href="../info/1048/50992.htm" class="publine">
  <div class="date"><span>2026-09-12</span></div>
  <div class="info"><h3>标题</h3><p>摘要…</p></div>
</a>
```


In [7]:
from urllib.parse import urljoin  # 相对路径 ../info/... 拼成完整网址
from bs4 import BeautifulSoup

# resp.text 是解码后的 HTML 字符串；html.parser 是 Python 自带的解析器
soup = BeautifulSoup(resp.text, "html.parser")
rows = []
# 只选新闻条目的 a.publine，不要 select("a")（导航栏也会被抓进来）
for a in soup.select("a.publine"):
    href = a.get("href")  # 属性用 .get；没有 href 就跳过
    if not href:
        continue
    title = a.select_one("h3")          # 这一条里的标题标签
    date = a.select_one(".date span")   # class="date" 里面的日期
    rows.append({
        "date": date.get_text(strip=True) if date else "",    # 缺标签就空字符串
        "title": title.get_text(strip=True) if title else "",
        "url": urljoin(URL, href),  # 底必须是当前列表页 URL，不要手搓
    })

print(len(rows))   # 首页大约 10 条
print(rows[0])


10
{'date': '2026-06-24', 'title': 'æ\x95\x99è\x82²é\x83¨å\x85\x9aç»\x84ä¹¦è®°ã\x80\x81é\x83¨é\x95¿æ\x80\x80è¿\x9bé¹\x8fè°\x83ç\xa0\x94è¥¿å\x8d\x97è´¢ç»\x8få¤§å\xad¦', 'url': 'https://www.swufe.edu.cn/info/1048/49802.htm'}


当堂过关：能打印出大约 10 条（首页一页的条数），第一条有日期和标题。导航栏里也有链接，所以**只选 `a.publine`**，不要 `soup.select("a")` 一把抓。


## 4. 再深一点：一篇正文

列表页只有摘要。正文在 `https://www.swufe.edu.cn/info/1048/xxxxx.htm`，正文容器是 `.v_news_content`。


In [ ]:
import time

def fetch_article(url):
    r = requests.get(url, headers=headers, timeout=15)
    r.encoding = "utf-8"
    s = BeautifulSoup(r.text, "html.parser")
    h = s.select_one(".art-tit h3")
    body = s.select_one(".v_news_content")
    return {
        "url": url,
        "title": h.get_text(strip=True) if h else "",
        "text": body.get_text("\n", strip=True) if body else "",
    }

sample = fetch_article(rows[0]["url"])
print(sample["title"])
print(sample["text"][:200])


当堂只拉 **1 篇**。确认有完整段落，再谈翻页。


## 课后练习题分享

- 用opencode写代码，存储为py代码
- 下载西财所有要闻